In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
#Importando o dataframe que será utilizado

df = pd.read_csv('Covid Data.csv')

In [3]:
#Exibindo o dataframe

df.head()

,USMER,MEDICAL_UNIT,SEX,PATIENT_TYPE,DATE_DIED,INTUBED,PNEUMONIA,AGE,PREGNANT,DIABETES,...,ASTHMA,INMSUPR,HIPERTENSION,OTHER_DISEASE,CARDIOVASCULAR,OBESITY,RENAL_CHRONIC,TOBACCO,CLASIFFICATION_FINAL,ICU
0,2,1,1,1,03/05/2020,97,1,65,2,2,...,2,2,1,2,2,2,2,2,3,97
1,2,1,2,1,03/06/2020,97,1,72,97,2,...,2,2,1,2,2,1,1,2,5,97
2,2,1,2,2,09/06/2020,1,2,55,97,1,...,2,2,2,2,2,2,2,2,3,2
3,2,1,1,1,12/06/2020,97,2,53,2,2,...,2,2,2,2,2,2,2,2,7,97
4,2,1,2,1,21/06/2020,97,2,68,97,1,...,2,2,1,2,2,2,2,2,3,97


In [4]:
#Verificando se existem valores ausentes

df.isnull().sum()

USMER                   0
MEDICAL_UNIT            0
SEX                     0
PATIENT_TYPE            0
DATE_DIED               0
INTUBED                 0
PNEUMONIA               0
AGE                     0
PREGNANT                0
DIABETES                0
COPD                    0
ASTHMA                  0
INMSUPR                 0
HIPERTENSION            0
OTHER_DISEASE           0
CARDIOVASCULAR          0
OBESITY                 0
RENAL_CHRONIC           0
TOBACCO                 0
CLASIFFICATION_FINAL    0
ICU                     0
dtype: int64

In [5]:
#Verificando se existem registros duplicados

df.duplicated().sum()

812049

In [6]:
#Verificando a quantidade de linhas e colunas

df.shape

(1048575, 21)

In [7]:
#Verificando os tipos das colunas

df.dtypes

USMER                    int64
MEDICAL_UNIT             int64
SEX                      int64
PATIENT_TYPE             int64
DATE_DIED               object
INTUBED                  int64
PNEUMONIA                int64
AGE                      int64
PREGNANT                 int64
DIABETES                 int64
COPD                     int64
ASTHMA                   int64
INMSUPR                  int64
HIPERTENSION             int64
OTHER_DISEASE            int64
CARDIOVASCULAR           int64
OBESITY                  int64
RENAL_CHRONIC            int64
TOBACCO                  int64
CLASIFFICATION_FINAL     int64
ICU                      int64
dtype: object

In [8]:
#Removendo colunas que não serão usadas

df.drop(df.columns[[4,5,8,20]], axis=1, inplace=True)

In [9]:
#Deixando o dataframe com 100 mil registros

df = df.head(100000)

In [10]:
#Ajustando valores em colunas específicas para garantir consistência nos dados

colunas_para_ajustar = [
    'DIABETES', 'COPD', 'ASTHMA', 'INMSUPR', 'HIPERTENSION',
    'OTHER_DISEASE', 'CARDIOVASCULAR', 'OBESITY', 'RENAL_CHRONIC', 'TOBACCO'
]
for col in colunas_para_ajustar:
    df[col] = df[col].replace(98, 2)


df['PNEUMONIA'] = df['PNEUMONIA'].replace(99, 2)


df['CLASIFFICATION_FINAL'] = np.where(df['CLASIFFICATION_FINAL'].isin([1,2,3]), 1, 0)

In [11]:
#Exibindo o dataframe final

df.head()

,USMER,MEDICAL_UNIT,SEX,PATIENT_TYPE,PNEUMONIA,AGE,DIABETES,COPD,ASTHMA,INMSUPR,HIPERTENSION,OTHER_DISEASE,CARDIOVASCULAR,OBESITY,RENAL_CHRONIC,TOBACCO,CLASIFFICATION_FINAL
0,2,1,1,1,1,65,2,2,2,2,1,2,2,2,2,2,1
1,2,1,2,1,1,72,2,2,2,2,1,2,2,1,1,2,0
2,2,1,2,2,2,55,1,2,2,2,2,2,2,2,2,2,1
3,2,1,1,1,2,53,2,2,2,2,2,2,2,2,2,2,0
4,2,1,2,1,2,68,1,2,2,2,1,2,2,2,2,2,1


In [12]:
#Criando variável de features

x = df.iloc[:, 0:16].values

In [13]:
#Criando variável de target

y = df.iloc[:, 16].values

In [14]:
#Dividindo os dados em treino(80%) e teste(20%)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [15]:
#Criando arquivo para salvar as variáveis de treino e teste

with open('variaveis_train_test.pkl', mode='wb') as f:
    pickle.dump([x_train, y_train, x_test, y_test], f)

In [16]:
# Criando o modelo KNN com 9 vizinhos

knn = KNeighborsClassifier(n_neighbors=9)

In [17]:
#Treinando o modelo

knn.fit(x_train, y_train)

KNeighborsClassifier(n_neighbors=9)

In [18]:
#Criando as previsões do modelo

previsoes = knn.predict(x_test)

c:\Users\gabri\anaconda3\lib\site-packages\sklearn\neighbors\_classification.py:228: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode, _ = stats.mode(_y[neigh_ind, k], axis=1)


In [19]:
#Exibindo a acuracidade do modelo

accuracy_score(y_test, previsoes)

0.78735

In [20]:
#Exibindo e analisando as demais métricas do modelo

print(classification_report(y_test, previsoes))

              precision    recall  f1-score   support

           0       0.57      0.35      0.43      4622
           1       0.82      0.92      0.87     15378

    accuracy                           0.79     20000
   macro avg       0.69      0.63      0.65     20000
weighted avg       0.76      0.79      0.77     20000



In [21]:
#Exibindo e analisando a matriz de confusão

print(confusion_matrix(y_test, previsoes))

[[ 1596  3026]
 [ 1227 14151]]
